## 1) Imports + locate NV2_array

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

def find_workspace_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'qua-libs').exists():
            return p
    raise RuntimeError('Could not find workspace root containing qua-libs')

WORKSPACE = find_workspace_root(Path.cwd())
ANALYSIS = (WORKSPACE / 'qua-libs/Quantum-Control-Applications/Optically addressable spin qubits/NV2_array/analysis').resolve()
NV2 = ANALYSIS.parent
DATA_ROOT = NV2 / 'Data'
sys.path.insert(0, str(ANALYSIS))
from nv2_analysis.dataset import DatasetReader  # noqa: E402
from nv2_analysis.fits import ExperimentFitter  # noqa: E402
plt.rcParams['figure.figsize'] = (9, 5)

## 2) Load saved arrays

In [ ]:
reader = DatasetReader(nv_root=NV2, data_root=DATA_ROOT)
EXPERIMENT_TAG = 'odmr'  # matches folder names like '#1_cw_odmr_...' or '#24_pulsed_odmr_...'
# Choose a specific dataset folder or leave None for latest matching tag:
# DATASET = r'2025-12-31\\#1_cw_odmr_122033'
# DATASET = r'C:\\\\...\\\\NV2_array\\\\Data\\\\2025-12-31\\\\#1_cw_odmr_122033'
DATASET = None

def find_latest_by_tag(tag: str) -> Path:
    candidates = [p.parent for p in DATA_ROOT.rglob('data.json') if p.is_file() and tag in p.parent.name.lower()]
    if not candidates:
        raise FileNotFoundError(f"No datasets matching tag {tag!r} under {DATA_ROOT}")
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

ds_folder = find_latest_by_tag(EXPERIMENT_TAG) if DATASET is None else reader.resolve_dataset(DATASET).folder
ds = reader.resolve_dataset(ds_folder)
data = reader.load(ds)

def pick_key(d: dict, keys: list[str]) -> str:
    for k in keys:
        if k in d:
            return k
    raise KeyError(f"None of these keys found: {keys}. Available keys: {sorted(d.keys())}")

k_counts = pick_key(data, ['counts_data', 'counts'])
k_ref = pick_key(data, ['counts_ref_data', 'counts_dark_data', 'counts_ref', 'counts_dark'])
k_mw = None
for candidate in ['MW_frequencies', 'mw_frequencies', 'mw_freq', 'frequencies', 'f_mw_hz']:
    if candidate in data:
        k_mw = candidate
        break
k_if = None
if k_mw is None:
    k_if = pick_key(data, ['IF_frequencies', 'if_frequencies', 'IF', 'if'])

counts_data = np.asarray(data[k_counts], dtype=float)
counts_ref_data = np.asarray(data[k_ref], dtype=float)

cfg = data.get('config') or {}
if k_mw is not None:
    f_mw_hz = np.asarray(data[k_mw], dtype=float)
    if_hz = None
    lo_hz = None
else:
    if_hz = np.asarray(data[k_if], dtype=float)
    try:
        lo_hz = float(cfg['elements']['NV']['mixInputs']['lo_frequency'])
    except Exception:
        lo_hz = None
    f_mw_hz = (lo_hz + if_hz) if lo_hz is not None else if_hz

f_mw_mhz = np.asarray(f_mw_hz, dtype=float) / 1e6

def get_readout_len_ns(cfg: dict) -> float | None:
    pulses = (cfg.get('pulses') or {})
    for k in ('long_readout_pulse_1', 'long_readout_pulse_2', 'readout_pulse_1', 'readout_pulse_2'):
        try:
            return float(pulses[k]['length'])
        except Exception:
            pass
    return None

readout_len_ns = get_readout_len_ns(cfg)
meas_len_s = (readout_len_ns * 1e-9) if readout_len_ns is not None else None

if meas_len_s is None:
    signal = counts_data
    ref = counts_ref_data
    y_unit = 'counts'
else:
    signal = counts_data / 1000.0 / meas_len_s
    ref = counts_ref_data / 1000.0 / meas_len_s
    y_unit = 'kcps'

with np.errstate(divide='ignore', invalid='ignore'):
    contrast = (ref - signal) / ref
    dip = 1.0 - contrast  # = signal/ref

print(f"Loaded: {ds.folder}")

## 3) Plot original arrays

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(f_mw_mhz, signal, 'o-', ms=3, label='signal')
ax[0].plot(f_mw_mhz, ref, 'o-', ms=3, label='reference')
ax[0].set_xlabel('MW frequency (MHz)')
ax[0].set_ylabel(y_unit)
ax[0].set_title('Raw data')
ax[0].grid(True, alpha=0.3)
ax[0].legend()
ax[1].plot(f_mw_mhz, dip, 'o-', ms=3)
ax[1].set_xlabel('MW frequency (MHz)')
ax[1].set_ylabel('signal/ref (dip)')
ax[1].set_title('Normalized ODMR')
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4) Fit

In [ ]:
from importlib import reload
import nv2_analysis.fits as fits
reload(fits)
fitter = fits.ExperimentFitter()

fit_res = fitter.fit_cw_odmr({
    'MW_frequencies': f_mw_hz,
    'IF_frequencies': if_hz if if_hz is not None else f_mw_hz,
    'counts_data': counts_data,
    'counts_ref_data': counts_ref_data,
    'config': cfg,
})

In [ ]:
# Plot first
x_hz = getattr(fit_res, 'x', None)
y_contrast = getattr(fit_res, 'y', None)
y_fit = getattr(fit_res, 'y_fit', None)
if x_hz is None:
    x_hz = f_mw_hz
if y_contrast is None:
    y_contrast = contrast
x_mhz = np.asarray(x_hz, dtype=float) / 1e6
y_dip = 1.0 - np.asarray(y_contrast, dtype=float)

plt.figure(figsize=(9, 4))
plt.plot(x_mhz, y_dip, 'o', ms=3, label='normalized signal')
if y_fit is not None:
    plt.plot(x_mhz, 1.0 - np.asarray(y_fit, dtype=float), '-', lw=2, label='fit')
plt.xlabel('MW frequency (MHz)')
plt.ylabel('signal/ref (dip)')
plt.title('ODMR fit')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Then print fit details
print('Fit results:')
print(f"- f0: {fit_res.params['f0_mw_mhz']:.6f} MHz")
if fit_res.params.get('suggested_NV_IF_freq_mhz') is not None:
    print(f"- suggested NV IF: {fit_res.params['suggested_NV_IF_freq_mhz']:.6f} MHz")
print(f"- gamma: {fit_res.params['gamma_mhz']:.6f} MHz")
print(f"- depth: {fit_res.params['depth']:.6g}")